# Evaluating and Observing Agents: Outcomes, Trajectories, Traces and Spans

The runnable code of [Evaluating and Observing Agents: Outcomes, Trajectories, Traces and Spans](https://residual-stream.lvtan71.workers.dev/posts/evaluating-and-observing-agents/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/evaluating-and-observing-agents.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "evaluating-and-observing-agents"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## pass@k and pass^k

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/evaluating-and-observing-agents/#passk-and-passk)

In [ ]:
from math import comb

# pass@k: at least one of k attempts succeeds (what a benchmark with
# retries, or a human picking the best of k, cares about). pass^k: all k
# succeed (what a user who asks k times, or k customers, cares about).
# For one task with per-attempt success p: 1 - (1 - p)^k and p^k.
print(f"{'p':>5s} " + " ".join(f"{'pass@' + str(k):>8s}" for k in (1, 4, 8)) + "   " +
      " ".join(f"{'pass^' + str(k):>8s}" for k in (1, 4, 8)))
for p in (0.5, 0.8, 0.9, 0.95, 0.99):
    print(f"{p:5.2f} " + " ".join(f"{1 - (1 - p) ** k:8.3f}" for k in (1, 4, 8)) + "   " +
          " ".join(f"{p ** k:8.3f}" for k in (1, 4, 8)))

# From n attempts with c successes, unbiased estimates for k <= n
# (Chen et al. 2021 for pass@k; the same counting gives pass^k).
def pass_at(n, c, k): return 1 - comb(n - c, k) / comb(n, k)
def pass_hat(n, c, k): return comb(c, k) / comb(n, k)
print(f"\n5 attempts, 3 succeed: pass@1 {pass_at(5, 3, 1):.2f}, pass@3 {pass_at(5, 3, 3):.2f}, "
      f"pass^3 {pass_hat(5, 3, 3):.2f}")

The post printed:

```text
    p   pass@1   pass@4   pass@8     pass^1   pass^4   pass^8
 0.50    0.500    0.938    0.996      0.500    0.062    0.004
 0.80    0.800    0.998    1.000      0.800    0.410    0.168
 0.90    0.900    1.000    1.000      0.900    0.656    0.430
 0.95    0.950    1.000    1.000      0.950    0.815    0.663
 0.99    0.990    1.000    1.000      0.990    0.961    0.923

5 attempts, 3 succeed: pass@1 0.60, pass@3 1.00, pass^3 0.10
```

## An agent, traced and evaluated

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/evaluating-and-observing-agents/#an-agent-traced-and-evaluated)

In [ ]:
import json
import random
import re
import time
import uuid
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# A small agent, traced. Qwen2.5-1.5B-Instruct with one tool, a calculator,
# on 20 word problems that each need two arithmetic steps on 3- and 4-digit
# numbers. Each problem is run 5 times (temperature 0.7, and the model's
# default repetition penalty of 1.1, stated explicitly). Every model call
# and tool call is recorded as a span-shaped record with OpenTelemetry's
# GenAI attribute names, including the opt-in message contents and each
# call's finish reason; the evaluation reads nothing else but the answer key.
PATH = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                         revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306")
MODEL = "qwen2.5-1.5b-instruct"
tok = AutoTokenizer.from_pretrained(PATH)
model = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float16).to(GPU).eval()
TOOLS = [{"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic "
          "expression with + - * / and parentheses, and return the number.",
          "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                         "required": ["expression"]}}}]

def calculator(expression):
    if not re.fullmatch(r"[\d\s+\-*/().]+", expression):
        return "error: only numbers and + - * / ( ) are allowed"
    return str(round(eval(expression), 6))                    # safe: the regex admits arithmetic only

rng = random.Random(0)
tasks = []
for _ in range(20):
    a, b, c, d = rng.randint(1000, 9999), rng.randint(100, 999), rng.randint(3, 9), rng.randint(100, 999)
    tasks.append((f"A warehouse holds {a} boxes. It ships out {b} boxes, then receives {c} deliveries of "
                  f"{d} boxes each. How many boxes does it hold now?", a - b + c * d))

def span(trace, name, parent, **attrs):
    s = {"trace_id": trace["id"], "span_id": uuid.uuid4().hex[:16], "parent_id": parent, "name": name,
         "start": time.perf_counter(), "attributes": attrs}
    trace["spans"].append(s)
    return s

def run(question, seed, max_turns=5):
    torch.manual_seed(seed)
    trace = {"id": uuid.uuid4().hex, "spans": []}
    root = span(trace, "invoke_agent warehouse-helper", None, **{"gen_ai.operation.name": "invoke_agent",
                "gen_ai.agent.name": "warehouse-helper"})
    messages = [{"role": "system", "content": "Use the calculator tool for every arithmetic step. "
                 "End with: Answer: <number>"}, {"role": "user", "content": question}]
    answer = lenient = final = None
    for _ in range(max_turns):
        prompt = tok.apply_chat_template(messages, tools=TOOLS, add_generation_prompt=True, tokenize=False)
        ids = tok(prompt, return_tensors="pt").input_ids.to(GPU)
        s = span(trace, f"chat {MODEL}", root["span_id"], **{"gen_ai.operation.name": "chat",
                 "gen_ai.provider.name": "local", "gen_ai.request.model": MODEL, "gen_ai.request.temperature": 0.7,
                 "gen_ai.request.max_tokens": 512})
        out = model.generate(ids, max_new_tokens=512, do_sample=True, temperature=0.7, top_p=1.0, top_k=0,
                             repetition_penalty=1.1)
        new = out[0, ids.shape[1]:]
        text = tok.decode(new, skip_special_tokens=True)
        s["end"] = time.perf_counter()
        stopped = new[-1].item() in (tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>"))
        s["attributes"].update({"gen_ai.usage.input_tokens": ids.shape[1], "gen_ai.usage.output_tokens": len(new),
                                "gen_ai.response.finish_reasons": ["stop" if stopped else "length"],
                                "gen_ai.output.messages": [{"role": "assistant", "content": text}]})
        calls = re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)
        messages.append({"role": "assistant", "content": text})
        if not calls:
            final = text
            m = re.search(r"Answer:\s*\$?(-?[\d,]+)", text)               # the format we asked for
            answer = int(m.group(1).replace(",", "")) if m else None
            nums = re.findall(r"-?\d[\d,]*", text)                        # or just the last number in the reply
            lenient = int(nums[-1].replace(",", "")) if nums else None
            break
        for c in calls:
            try:
                args = json.loads(c)["arguments"]; expr = args["expression"]
            except (ValueError, KeyError, TypeError):
                args, expr = c, None
            # the model emits no call id, so the harness assigns one
            t = span(trace, "execute_tool calculator", root["span_id"], **{"gen_ai.operation.name": "execute_tool",
                     "gen_ai.tool.name": "calculator", "gen_ai.tool.call.id": uuid.uuid4().hex[:8],
                     "gen_ai.tool.call.arguments": args})
            result = calculator(expr) if expr is not None else "error: malformed tool call"
            t["end"] = time.perf_counter()
            if result.startswith("error"):
                t["attributes"]["error.type"] = "invalid_arguments"; t["status"] = "ERROR"
            else:
                t["attributes"]["gen_ai.tool.call.result"] = result
            messages.append({"role": "tool", "content": result})
    root["end"] = time.perf_counter()
    return trace, answer, lenient, final

results = []
for i, (q, truth) in enumerate(tasks):
    for k in range(5):
        trace, ans, lenient, final = run(q, seed=100 * i + k)
        results.append({"task": i, "truth": truth, "answer": ans, "lenient": lenient, "final": final, "trace": trace})
    print(f"task {i:2d}: {sum(r['answer'] == truth for r in results[-5:])}/5 correct as asked, "
          f"{sum(r['lenient'] == truth for r in results[-5:])}/5 by the last number", flush=True)
json.dump(results, open("traces.json", "w"))

In [ ]:
import json
import re
import numpy as np
from math import comb

# Evaluating the agent from its traces: every fact below is read from the
# span records (message contents, finish reasons, tool calls); the only
# other input is the answer key.
results = json.load(open("traces.json"))
key = {r["task"]: r["truth"] for r in results}

def facts(r):
    spans = r["trace"]["spans"]
    chats = [s["attributes"] for s in spans if s["attributes"]["gen_ai.operation.name"] == "chat"]
    tools = [s["attributes"] for s in spans if s["attributes"]["gen_ai.operation.name"] == "execute_tool"]
    final = chats[-1]["gen_ai.output.messages"][0]["content"]
    strict = re.search(r"Answer:\s*\$?(-?[\d,]+)", final)
    nums = re.findall(r"-?\d[\d,]*", final)
    return {"final": final, "tools": tools,
            "truncated": any(c["gen_ai.response.finish_reasons"] == ["length"] for c in chats),
            "strict": int(strict.group(1).replace(",", "")) if strict else None,
            "lenient": int(nums[-1].replace(",", "")) if nums else None,
            "tool_has_truth": str(key[r["task"]]) in [t.get("gen_ai.tool.call.result", "").split(".")[0] for t in tools]}
F = [(r, facts(r)) for r in results]

pass_at = lambda n, c, k: 1 - comb(n - c, k) / comb(n, k)
pass_hat = lambda n, c, k: comb(c, k) / comb(n, k)
for name, field in (("as asked ('Answer: <number>')", "strict"), ("by the last number in the reply", "lenient")):
    per = {}
    for r, f in F:
        per.setdefault(r["task"], []).append(f[field] == key[r["task"]])
    print(f"outcome scored {name}, 20 tasks x 5 runs:")
    for k in (1, 3, 5):
        print(f"  pass@{k} {np.mean([pass_at(5, sum(v), k) for v in per.values()]):.2f}   "
              f"pass^{k} {np.mean([pass_hat(5, sum(v), k) for v in per.values()]):.2f}")
    print(f"  tasks solved in all 5 runs: {sum(sum(v) == 5 for v in per.values())} of 20")

def trajectory(f, truth):
    right = f["lenient"] == truth
    if f["truncated"]:
        return "cut off by the token budget" + (" (still right)" if right else "")
    if right:
        return "right, the calculator produced the answer" if f["tool_has_truth"] else "right, the calculator didn't produce it"
    if f["tools"] and not f["tool_has_truth"]:
        return "wrong: asked the calculator the wrong question"
    if f["tool_has_truth"]:
        return "wrong: calculator had it, reply differs"
    return "wrong: no calculator"
cats = {}
for r, f in F:
    c = trajectory(f, key[r["task"]]); cats[c] = cats.get(c, 0) + 1
print("\ntrajectories (100 runs):")
for c, v in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {v:3d}  {c}")
calls = [len(f["tools"]) for _, f in F]
print(f"  calculator calls per run: none {calls.count(0)}, one {calls.count(1)}, two or more {sum(c >= 2 for c in calls)}"
      f" (the instructions asked for one per arithmetic step: two)")
print(f"  runs with no 'Answer:' line: {sum(f['strict'] is None for _, f in F)}; of those, replies saying "
      f"'the answer is': {sum(f['strict'] is None and 'answer is' in f['final'].lower() for _, f in F)}")

spans = [s for r in results for s in r["trace"]["spans"]]
by = lambda op: [s for s in spans if s["attributes"]["gen_ai.operation.name"] == op]
dur = lambda ss: np.array([s["end"] - s["start"] for s in ss])
chat, tool, agents = by("chat"), by("execute_tool"), by("invoke_agent")
tin = sum(s["attributes"]["gen_ai.usage.input_tokens"] for s in chat)
tout = sum(s["attributes"]["gen_ai.usage.output_tokens"] for s in chat)
right = sum(f["lenient"] == key[r["task"]] for r, f in F)
print(f"\ndashboard: {len(agents)} runs, {len(chat)} model calls, {len(tool)} tool calls")
print(f"  per run: {len(chat) / len(agents):.1f} model calls, {tin / len(agents):,.0f} tokens in, {tout / len(agents):.0f} out; "
      f"per successful run: {(tin + tout) / right:,.0f} tokens")
print(f"  run latency p50 {np.median(dur(agents)):.1f} s, p95 {np.percentile(dur(agents), 95):.1f} s; "
      f"model call p50 {np.median(dur(chat)):.2f} s; tool call p50 {np.median(dur(tool)) * 1e6:.0f} us")

r, f = next((r, f) for r, f in F if trajectory(f, key[r["task"]]).startswith("wrong: asked"))
print(f"\none trace (task {r['task']}, right answer {key[r['task']]}):")
t0 = r["trace"]["spans"][0]["start"]
for s in r["trace"]["spans"]:
    a = s["attributes"]
    extra = (f"in {a['gen_ai.usage.input_tokens']} out {a['gen_ai.usage.output_tokens']} tokens, "
             f"finish {a['gen_ai.response.finish_reasons'][0]}" if a["gen_ai.operation.name"] == "chat" else
             f"{a.get('gen_ai.tool.call.arguments')} -> {a.get('gen_ai.tool.call.result')}"
             if a["gen_ai.operation.name"] == "execute_tool" else "")
    print(f"{'  ' if s['parent_id'] is None else '      '}{s['name']:38s} {(s['start'] - t0) * 1e3:7.0f} ms  "
          f"+{(s['end'] - s['start']) * 1e3:6.0f} ms  {extra}")
print(f"  final reply: {f['final']!r}")

The post printed:

```text
outcome scored as asked ('Answer: <number>'), 20 tasks x 5 runs:
  pass@1 0.15   pass^1 0.15
  pass@3 0.39   pass^3 0.00
  pass@5 0.55   pass^5 0.00
  tasks solved in all 5 runs: 0 of 20
outcome scored by the last number in the reply, 20 tasks x 5 runs:
  pass@1 0.93   pass^1 0.93
  pass@3 1.00   pass^3 0.81
  pass@5 1.00   pass^5 0.70
  tasks solved in all 5 runs: 14 of 20

trajectories (100 runs):
   81  right, the calculator produced the answer
   11  right, the calculator didn't produce it
    4  wrong: no calculator
    3  wrong: asked the calculator the wrong question
    1  cut off by the token budget (still right)
  calculator calls per run: none 14, one 84, two or more 2 (the instructions asked for one per arithmetic step: two)
  runs with no 'Answer:' line: 83; of those, replies saying 'the answer is': 19

dashboard: 100 runs, 186 model calls, 88 tool calls
  per run: 1.9 model calls, 454 tokens in, 81 out; per successful run: 575 tokens
  run latency p50 2.5 s, p95 10.9 s; model call p50 1.55 s; tool call p50 33 us

one trace (task 9, right answer 12975):
  invoke_agent warehouse-helper                0 ms  +  2468 ms  
      chat qwen2.5-1.5b-instruct                   1 ms  +  1594 ms  in 213 out 33 tokens, finish stop
      execute_tool calculator                   1595 ms  +     0 ms  {'expression': '8113-(423+7*755)'} -> 2405
      chat qwen2.5-1.5b-instruct                1596 ms  +   872 ms  in 268 out 12 tokens, finish stop
  final reply: 'The warehouse holds 2405 boxes now.'
```